Notebook illustrating the STA-LTA mining method of the paper. 

We only consider data from ILL11 for 2018. 

Data can be downloaded from: https://doi.org/10.12686/SED/NETWORKS/XP

In [1]:
import os
import numpy as np
import obspy
import pandas as pd

from datamod.loading_utils import (
    preproc_flow_annotations,
    find_date_in_strings,
    remove_duplicate_traces,
    remove_overlaps,
)
from datamod.preproc_utils import preproc_stream
from sta_lta.utils import slta_compute_iou, slta_detections_from_paths
from datetime import timedelta

C:\Users\franc\AppData\Local\Temp\ipykernel_32952\1835586087.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
folder = "../data/XP/2018/ILL11/HHZ.D/"
stream_paths = [folder + f for f in os.listdir(folder) if not f.startswith("._")]
stream_paths = np.array(sorted(stream_paths, key=lambda f: int(f.rsplit(".", 1)[-1])))

flows = preproc_flow_annotations(pd.read_csv("../catalogues/flow_catalogue.csv"))
flows = flows[flows["station"] == "ILL11"].reset_index(drop=True)
flows = flows[flows["stop"] < obspy.UTCDateTime("2018-12-31T23:59:59")].reset_index(
    drop=True
)

high_conf_flows = flows[flows["confidence"] == 1.0].reset_index(drop=True)
lower_conf_flows = flows[flows["confidence"] != 1.0].reset_index(drop=True)

As in the case of the IF trigger, we calibrate the STA-LTA trigger by focusing only on days with high-confidence segments.

Note. As far as is feasible (i.e. available data and no breakages) we append append the seismic waveform of the previous day so that the STA-LTA can immediately compute its characteristic function. 

In [3]:
# in cases where a high-confidence segment spans two calendar days.
start_time_days = np.array([str(i)[:10] for i in high_conf_flows["start"]])
stop_time_days = np.array([str(i)[:10] for i in high_conf_flows["stop"]])
flow_days = np.unique(np.append(start_time_days, stop_time_days))

prev_days = []

for day in flow_days:
    prev_days.append(str(pd.to_datetime(day) - timedelta(days=1))[:10])

prev_days = np.array(prev_days)

print(flow_days, prev_days)

['2018-06-12' '2018-07-25' '2018-08-08'] ['2018-06-11' '2018-07-24' '2018-08-07']


Read in the streams, including previous days.

In [4]:
high_conf_st = []

for i in range(flow_days.shape[0]):
    flow_st = obspy.read(find_date_in_strings(stream_paths, flow_days[i])[0])
    flow_st = remove_duplicate_traces(flow_st)
    flow_st = remove_overlaps(flow_st)
    preproc_stream(flow_st)

    prev_day_st = obspy.read(find_date_in_strings(stream_paths, prev_days[i])[0])
    prev_day_st = remove_duplicate_traces(prev_day_st)
    prev_day_st = remove_overlaps(prev_day_st)
    preproc_stream(prev_day_st)

    high_conf_st.append({"res_tr": prev_day_st[-1], "st": flow_st})

Perform the local grid searches.

In [5]:
sampling_rate = 100
lw_grid = 5000 * np.power(2.0, np.arange(-5, 6))
sw_grid = 0.003125 * np.power(2.0, np.arange(11))
onset_grid = 0.1875 * np.power(2.0, np.arange(11))
offset_grid = 0.00390625 * np.power(2.0, np.arange(11))

iou_table = np.zeros([11, 11, 11, 11])
iou_table[:] = np.nan
ic, jc, kc, lc = 5, 5, 5, 5

stop = False

In [6]:
try:
    iou_table = np.load("../output/sta_lta_grids/ILL11_illustration.npy")

except (FileNotFoundError, OSError):
    while not stop:
        stop = True
        for i in range(max(0, ic - 1), min(10, ic + 2)):
            for j in range(max(0, jc - 1), min(10, jc + 2)):
                for k in range(max(0, kc - 1), min(10, kc + 2)):
                    for l in range(max(0, lc - 1), min(10, lc + 2)):
                        if not np.isnan(iou_table[i, j, k, l]):
                            continue

                        stop = False
                        lw = int(sampling_rate * lw_grid[j])
                        sw = sampling_rate * sw_grid[i] * lw_grid[j]
                        onset_thres, offset_thres = onset_grid[k], offset_grid[l]

                        iou_table[i, j, k, l] = slta_compute_iou(
                            high_conf_st,
                            high_conf_flows,
                            sw,
                            lw,
                            onset_thres,
                            offset_thres,
                        )

                        # print(i,j,k,l)
        # all the positions around the current one has been filled.
        if stop:
            break

        temp_table = iou_table.copy()
        temp_table[np.isnan(temp_table)] = 0

        # check if we found a better solution than the current...otherwise stop
        if np.max(temp_table) > iou_table[ic, jc, kc, lc]:
            stop = False
            ic, jc, kc, lc = np.unravel_index(np.argmax(temp_table), temp_table.shape)

        else:
            stop = True

        np.save("../output/sta_lta_grids/ILL11_illustration.npy", iou_table)

In [8]:
iou_table[np.isnan(iou_table)] = 0
iou_table[np.isnan(iou_table)] = 0
ic, jc, kc, lc = np.unravel_index(np.argmax(iou_table), iou_table.shape)

lw = int(sampling_rate * lw_grid[jc])
sw = sampling_rate * sw_grid[ic] * lw_grid[jc]
onset_thres, offset_thres = onset_grid[kc], offset_grid[lc]
print(sw, lw, onset_thres, offset_thres)

25000.0 500000 6.0 0.125


In [9]:
slta_detections = slta_detections_from_paths(
    stream_paths, sw, lw, onset_thres, offset_thres
)